

### Завдання 1: Виклик LLM з базовим промптом

**Мета:** навчитися викликати LLM через LangChain зі звичайним текстовим промптом.

**Що потрібно зробити:**

1. Створіть промпт, який дозволяє отримати інформацію простою мовою на тему "Квантові обчислення". Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

2. Обмежте відповідь до 200 символів і пропишіть в промпті, аби відповідь була короткою (це зекономить вам час і гроші на згенеровані токени).

3. Встановіть своє значення температури на власний розсуд (тут немає правильного чи неправильного значення) і напишіть коментарем, чому ви обрали саме таке значення для цього завдання.

**Вибір моделі:** можна скористатись як моделлю з HuggingFace, так і ChatGPT будь-якої версії, яка вам до вподоби і пасує за прайсингом. В обох випадках потрібно імпортувати відповідний клас з LangChain для виклику LLM за API.

**Мова запитів:** промпти можна писати як українською, так і англійською — орієнтуйтесь на те, де і для чого ви хочете потім використовувати цей проєкт. У розв'язках промпти — українською.

---

**🔐 Як безпечно зберігати і підвантажувати API-ключі**

API-токен потрібно зчитувати з безпечного джерела, а **не хардкодити в ноутбуці**. Якщо хтось отримає доступ до вашого ключа, він буде витрачати токени за ваш рахунок, а вам це не треба :)

Є кілька способів. Перший ми використовували на лекції, ще два для розширення вашого розуміння, як ще це можна зробити і що шлях не лише один. Для виконання цього ДЗ можете використовувати будь-який спосіб підвантаження ключів у ноутбук.

**Спосіб 1: Файл `creds.json` (рекомендований)**

Створіть файл `creds.json` з вашими ключами, завантажте його в Google Colab під час роботи, але **не здавайте** цей файл у ДЗ і **не комітьте** в git.

```python
import json
with open("creds.json") as f:
    creds = json.load(f)
api_key = creds["HF_TOKEN"]
```

**Спосіб 2: Google Colab Secrets**

У лівій панелі Colab натисніть іконку 🔑 (Secrets) → "Add new secret" → введіть назву (наприклад, `HF_TOKEN`) та значення ключа → увімкніть тогл доступу для ноутбука.

```python
from google.colab import userdata
api_key = userdata.get("HF_TOKEN")
```

Зручно тим, що ключ зберігається в акаунті і доступний у всіх ваших ноутбуках. Мінус — при кожній новій сесії потрібно перевірити, що доступ увімкнено.

**Спосіб 3: Google AI Studio (для Gemini API)**

Якщо працюєте з моделями Google Gemini, отримати безкоштовний API-ключ можна в [Google AI Studio](https://aistudio.google.com/app/apikey): увійдіть з Google-акаунтом → натисніть "Get API key" → "Create API key". Далі використовуйте ключ через будь-який із способів вище.



In [1]:
import json, os
from langchain_groq import ChatGroq

# Load key
with open("creds.json") as f:
    creds = json.load(f)

os.environ["GROQ_API_KEY"] = creds["GROQ_API_KEY"]

# LLM
llm = ChatGroq(
    api_key=os.environ["GROQ_API_KEY"],
    model="llama-3.1-8b-instant",
    temperature=0.3,
    max_tokens=200
)

prompt = """
Стисло поясни простими словами, що таке квантові обчислення.
Дай визначення, ключові переваги та поточні дослідження.
Стисла відповідь у 2–3 коротких реченнях.
"""

response = llm.invoke(prompt)
print(response)

content="Квантові обчислення - це нова техніка обробки інформації, яка використовує властивості квантової фізики для виконання обчислень. Її головна перевага - можливість обробляти дуже великі об'єкти даних дуже швидко, що може бути дуже корисно для багатьох галузей, зокрема кріогенної фізики, матеріалознавства та медицини." additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 85, 'total_tokens': 176, 'completion_time': 0.194997758, 'completion_tokens_details': None, 'prompt_time': 0.004726188, 'prompt_tokens_details': None, 'queue_time': 0.017742554, 'total_time': 0.199723946}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6a1eabf260', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d8236-0ba6-7752-ba69-c84ef74758eb-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 85, 'output_tokens': 91, 'total_tokens': 176}


Я обрала температуру 0.3, оскільки низька температура дає більш стабільні, передбачувані відповіді. Це важливо для стислого пояснення без зайвої креативності.

### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
import json, os

# Load key
with open("creds.json") as f:
    creds = json.load(f)

os.environ["GROQ_API_KEY"] = creds["GROQ_API_KEY"]

# LLM
llm = ChatGroq(
    api_key=os.environ["GROQ_API_KEY"],
    model="llama-3.1-8b-instant",
    temperature=0.3,
    max_tokens=300
)

# Prompt template
template = """
Поясни тему: "{topic}"
Поясни простими словами, у 3–4 реченнях.
Дай коротке, структуроване пояснення.
"""

prompt = PromptTemplate(
    input_variables=["topic"],
    template=template
)

topics = [
    "Баєсівські методи в машинному навчанні",
    "Трансформери в машинному навчанні",
    "Explainable AI"
]

for t in topics:
    final_prompt = prompt.format(topic=t)
    response = llm.invoke(final_prompt)
    print(f"\n=== Тема: {t} ===\n")
    print(response)
    print("\n" + "-"*80 + "\n")


=== Тема: Баєсівські методи в машинному навчанні ===

content="**Що таке Баєсівські методи в машинному навчанні?**\n\nБаєсівські методи в машинному навчанні — це набір технік, які використовують статистичні дані для навчання моделей передбачення. Вони ґрунтуються на ідеї Баєса про те, як об'єднати попередні знання зі спостережуваними даними, щоб зробити передбачення більш точними. Баєсівські методи застосовують такі техніки, як баєсова мережа, марковські мережі та гіберніанські мережі, щоб навчати моделі передбачати результати на основі попередніх даних.\n\n**Як працюють Баєсівські методи?**\n\n1. **Підготовка даних**: Збір та підготовка даних для навчання моделі.\n2. **Навчання моделі**: Використання Баєсівських методів для навчання моделі передбачення на основі підготовлених даних.\n3. **Представлення результатів**: Використання навченої моделі для передбачення результатів на основі нових даних.\n\n**НавANTI Баєсівських методів**\n\n- Баєсова мережа: використовує графічну репрезента



### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [5]:
!pip install -q langchain_community duckduckgo_search

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("Obama's first name?")

In [17]:
from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
import json, os

# Load Groq key
with open("creds.json") as f:
    creds = json.load(f)
os.environ["GROQ_API_KEY"] = creds["GROQ_API_KEY"]

# LLM
llm = ChatGroq(
    api_key=os.environ["GROQ_API_KEY"],
    model="llama-3.1-8b-instant",
    temperature=0.2
)

# Tool: DuckDuckGo search
search = DuckDuckGoSearchRun()

# Prompt for the agent
prompt = PromptTemplate.from_template("""
You are a scientific research extraction agent. Using ONLY the search results
below, extract up to 5 REAL scientific publications on artificial intelligence
published in 2024–2026.

Search results:
{search_results}

HARD RULES:
- Include ONLY real scientific papers (peer‑reviewed or preprints).
- MUST include authors. If authors are missing → SKIP the item.
- MUST include year. If year is missing → SKIP the item.
- NO news articles, NO blogs, NO conferences, NO tech reports, NO summaries.
- NO invented titles, NO invented authors, NO invented years.
- NO repetition.
- If fewer than 5 valid papers exist → return only the valid ones.

OUTPUT FORMAT (exactly):
1. Title:
   Authors:
   Year:
   Summary (1–2 sentences):
   URL:
""")

# Agent pipeline (LCEL)
agent = (
    {"search_results": search | RunnablePassthrough()}
    | prompt
    | llm
)

# Run agent
query = "latest scientific publications artificial intelligence"
result = agent.invoke(query)

print(result)

content='Based on the provided search results, here are 4 valid scientific publications on artificial intelligence:\n\n1. Reverse predictivity for bidirectional comparison of neural networks and biological brains\n   Authors: Sabine Muzellec et al\n   Year: 2026\n   Summary: This study compares neural networks and biological brains using reverse predictivity, a method that evaluates the ability of a model to predict previously unseen data. The results show that neural networks can be more efficient than biological brains in certain tasks.\n   URL: (Unfortunately, the URL is not provided in the search results, but the paper is published in Nature Machine Intelligence in 2026)\n\n2. ArtificialIntelligenceand the Wellbeing of Workers\n   Authors: (Authors not provided in the search results)\n   Year: 2024\n   Unfortunately, I cannot provide this paper as the authors are missing.\n\n3. Bringing BiomedicalArtificialIntelligenceinto Practice: Graph-Based Hypothesis Validation Using BioAssays



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [22]:
from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.prompts import PromptTemplate
import json, os, re

# -----------------------------
# 1. Load API key
# -----------------------------
with open("creds.json") as f:
    creds = json.load(f)

os.environ["GROQ_API_KEY"] = creds["GROQ_API_KEY"]

# -----------------------------
# 2. LLM
# -----------------------------
llm = ChatGroq(
    api_key=os.environ["GROQ_API_KEY"],
    model="llama-3.1-8b-instant",
    temperature=0.2,
    max_tokens=800
)

# -----------------------------
# 3. Search tool
# -----------------------------
search = DuckDuckGoSearchRun()

# -----------------------------
# 4. Extract sales history from text
# -----------------------------
def extract_sales_history(text):
    pattern = r"(20\d{2}).*?(\d+)\s*т"
    matches = re.findall(pattern, text)
    return [(int(y), float(t)) for y, t in matches]

# -----------------------------
# 5. Business analysis function
# -----------------------------
def run_business_agent(user_query):

    # Python: parse history
    history = extract_sales_history(user_query)

    # Internet: fetch external data
    weather_info = search.run(
        "Brazil orange harvest weather conditions 2024 2025 2026 impact on orange production"
    )
    inflation_info = search.run(
        "Brazil inflation rate 2024 2025 2026 forecast"
    )
    demand_info = search.run(
        "global orange demand and orange juice market outlook 2025 2026"
    )

    # Prompt
    prompt = PromptTemplate.from_template("""
Ти — бізнес-аналітик. На основі даних нижче зроби прогноз експорту апельсинів на 2026 рік.

Вхідні дані користувача:
{user_query}

Історія продажів (розпізнана Python):
{history}

Погодні умови Бразилії:
{weather_info}

Інфляція в Бразилії:
{inflation_info}

Глобальний попит:
{demand_info}

Завдання:
- Зроби прогноз експорту на 2026 рік (одне число + діапазон).
- Використай тренд історичних даних.
- Врахуй погоду, інфляцію та попит.
- Дай коротке бізнес-обґрунтування.
""")

    final_prompt = prompt.format(
        user_query=user_query,
        history=history,
        weather_info=weather_info,
        inflation_info=inflation_info,
        demand_info=demand_info
    )

    # Run LLM
    return llm.invoke(final_prompt)

# -----------------------------
# 6. Run example
# -----------------------------
user_query = """
Ми експортуємо апельсини з Бразилії. 
В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. 
Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 
враховуючи погодні умови в Бразилії і попит на апельсини в світі 
виходячи з економічної ситуації.
"""

result = run_business_agent(user_query)
print(result.content)

Згідно з історичними даними експорту апельсинів з Бразилії, я зроблю прогноз експорту на 2026 рік.

**Історія даних:**

[(2022, 200.0), (2023, 190.0), (2024, 210.0), (2025, 220.0)]

**Тренд історичних даних:**

За останні роки спостерігається зростання експорту апельсинів з Бразилії. У 2022 році експорт становив 200 тонн, у 2023 році - 190 тонн, у 2024 році - 210 тонн, у 2025 році - 220 тонн. Тренд зростання експорту становить близько 5% на рік.

**Погодні умови Бразилії:**

За останні роки погода в Бразилії була досить несприятливою для виробництва апельсинів. У 2022 році було зафіксовано найнижчий рівень виробництва апельсинів за останні 30 років. У 2025 році очікується зниження виробництва апельсинів на 24,36% порівняно з попереднім сезоном. Однак, згідно з прогнозами, виробництво апельсинів в Бразилії зросте до 292,94 мільйонів коробок у сезоні 2025-2026.

**Інфляція в Бразилії:**

Інфляція в Бразилії знижується протягом останніх років. У лютому 2026 року інфляція становила 4,14%, 

**Висновок:**

Агент:
- правильно працює
- використовує інтернет
- аналізує дані
- дає прогноз
- формує бізнес‑висновок
  
Але:
- математична частина могла б бути точнішою
- Python можна використати глибше
- можна додати сценарії та ризики
- можна покращити узгодженість зовнішніх даних


In [1]:
from typing import Annotated
import json, os

from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_experimental.utilities.python import PythonREPL
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage, ToolMessage

# -----------------------------
# 1. Load API key
# -----------------------------
with open("creds.json") as f:
    creds = json.load(f)
os.environ["GROQ_API_KEY"] = creds["GROQ_API_KEY"]

# -----------------------------
# 2. LLM
# -----------------------------
llm = ChatGroq(
    api_key=os.environ["GROQ_API_KEY"],
    model="qwen/qwen3-32b",
    temperature=0.2,
)

# -----------------------------
# 3. Tools
# -----------------------------
search = DuckDuckGoSearchRun()
python_repl = PythonREPL()

@tool
def python_repl_tool(code: Annotated[str, "Python code to execute"]):
    """Execute Python code inside REPL."""
    try:
        return python_repl.run(code)
    except Exception as e:
        return f"Python error: {e}"

@tool
def web_search(query: Annotated[str, "Search query"]):
    """Search the internet."""
    return search.run(query)

tools = {
    "python_repl_tool": python_repl_tool,
    "web_search": web_search,
}

# -----------------------------
# 4. Prompt
# -----------------------------
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a business analyst. Use python_repl_tool and web_search "
     "to compute regression and gather external data. Then produce a final forecast."
    ),
    ("human", "{input}")
])

llm_with_tools = llm.bind_tools(list(tools.values()))

# -----------------------------
# 5. TOOL LOOP (working)
# -----------------------------
def run_agent(user_query: str):
    messages = prompt.format_messages(input=user_query)

    while True:
        # 1) LLM THINKS
        response: AIMessage = llm_with_tools.invoke(messages)

        # 2) If no tool calls → FINAL ANSWER
        if not response.tool_calls:
            return response.content

        # 3) Add LLM message BEFORE executing tools
        messages.append(response)

        # 4) Execute tools
        for call in response.tool_calls:
            tool_name = call["name"]
            args = call["args"]

            tool_fn = tools[tool_name]
            tool_result = tool_fn.invoke(args)

            # 5) Add tool result as ToolMessage
            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=call["id"]
                )
            )

        # 6) LOOP CONTINUES — LLM will now see the tool result

# -----------------------------
# 6. Example run
# -----------------------------
user_query = """
Ми експортуємо апельсини з Бразилії. 
В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. 
Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 
враховуючи погодні умови в Бразилії і попит на апельсини в світі.
"""

result = run_agent(user_query)
print("\n=== FINAL ANSWER ===\n")
print(result)

Python REPL can execute arbitrary code. Use with caution.



=== FINAL ANSWER ===

The linear regression model predicts an export of **230 tons** for 2026 based on historical trends. However, external factors like weather and global demand should adjust this forecast:

1. **Weather in Brazil**: Recent dry conditions in 2025 may slightly reduce production, but no major droughts are reported for 2026.  
2. **Global Demand**: Stable demand for citrus fruits is observed, with no sharp declines expected.  

**Final Forecast**:  
Considering moderate weather and steady demand, a **235-ton export** in 2026 is reasonable (adjusted +2.5% from the model's prediction to account for slight production resilience).  

<final_answer>235</final_answer>


In [3]:
# =============================
# LangGraph + LangSmith version
# =============================

from typing import Annotated, TypedDict, List
import json, os

from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_experimental.utilities.python import PythonREPL
from langchain_core.tools import tool
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    ToolMessage,
    SystemMessage,
)
from langgraph.graph import StateGraph, END

# -----------------------------
# 1. Load API keys
# -----------------------------
with open("creds.json") as f:
    creds = json.load(f)

os.environ["GROQ_API_KEY"] = creds["GROQ_API_KEY"]
os.environ["LANGCHAIN_API_KEY"] = creds["LANGCHAIN_API_KEY"]
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "task4_orange_export_agent"

# -----------------------------
# 2. LLM
# -----------------------------
llm = ChatGroq(
    api_key=os.environ["GROQ_API_KEY"],
    model="qwen/qwen3-32b",
    temperature=0.2,
)

# -----------------------------
# 3. Tools
# -----------------------------
search = DuckDuckGoSearchRun()
python_repl = PythonREPL()

@tool
def python_repl_tool(code: Annotated[str, "Python code to execute"]):
    """Execute Python code inside REPL."""
    try:
        return python_repl.run(code)
    except Exception as e:
        return f"Python error: {e}"

@tool
def web_search(query: Annotated[str, "Search query"]):
    """Search the internet."""
    return search.run(query)

tools = {
    "python_repl_tool": python_repl_tool,
    "web_search": web_search,
}

llm_with_tools = llm.bind_tools(list(tools.values()))

# -----------------------------
# 4. LangGraph state
# -----------------------------
class AgentState(TypedDict):
    messages: List
    iterations: int

# -----------------------------
# 5. LLM node
# -----------------------------
def llm_node(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": state["messages"] + [response]}

# -----------------------------
# 6. Tool node
# -----------------------------
def tool_node(state: AgentState):
    last = state["messages"][-1]
    new_messages = state["messages"]

    for call in last.tool_calls:
        tool_name = call["name"]
        args = call["args"]

        tool_fn = tools[tool_name]
        result = tool_fn.invoke(args)

        new_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=call["id"],
            )
        )

    return {
        "messages": new_messages,
        "iterations": state["iterations"] + 1,
    }

# -----------------------------
# 7. Router: should continue?
# -----------------------------
def should_continue(state: AgentState):
    last = state["messages"][-1]

    # Stop if too many tool cycles
    if state["iterations"] >= 3:
        return "final"

    # If LLM wants to call a tool → go to tool node
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tool"

    # Otherwise → final answer
    return "final"

# -----------------------------
# 8. Final node
# -----------------------------
def final_node(state: AgentState):
    last = state["messages"][-1]
    return {"messages": [last]}

# -----------------------------
# 9. Build LangGraph
# -----------------------------
graph = StateGraph(AgentState)

graph.add_node("llm", llm_node)
graph.add_node("tool", tool_node)
graph.add_node("final", final_node)

graph.set_entry_point("llm")

graph.add_conditional_edges(
    "llm",
    should_continue,
    {
        "tool": "tool",
        "final": "final",
    },
)

graph.add_edge("tool", "llm")
graph.add_edge("final", END)

app = graph.compile()

# -----------------------------
# 10. Run example
# -----------------------------
system_msg = SystemMessage(
    content=(
        "You are a business analyst. Use python_repl_tool and web_search "
        "to compute regression on the provided export data and gather "
        "external information (weather in Brazil, global orange demand). "
        "Then produce a clear final forecast for 2026 in tons, with a short justification."
    )
)

user_query = """
Ми експортуємо апельсини з Бразилії. 
В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. 
Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026
враховуючи погодні умови в Бразилії і попит на апельсини в світі.
"""

initial_messages = [
    system_msg,
    HumanMessage(content=user_query),
]

result = app.invoke({
    "messages": initial_messages,
    "iterations": 0,
})

print("\n=== FINAL ANSWER ===\n")
print(result["messages"][-1].content)


=== FINAL ANSWER ===

**Final Forecast for 2026 Orange Exports from Brazil: 225–230 tons**

**Justification:**  
1. **Baseline Trend:** Linear regression on historical data (2022–2025) shows a steady increase of **+8 tons/year**, projecting **225 tons** for 2026.  
2. **Weather Impact:** No extreme weather events were reported in Brazil for 2026 (based on available data), suggesting stable production conditions.  
3. **Global Demand:** While long-term demand forecasts (up to 2034) indicate growth, no significant disruptions or surges are expected in 2026.  

The forecast assumes moderate weather and stable demand, aligning with the observed trend. If major climatic or market shifts occur (e.g., droughts, trade policy changes), adjustments may be needed.


**Висновок**
Отримана відповідь агента є логічною та зрозумілою, проте аналіз показує, що модель не повністю виконала поставлене завдання. Зокрема, агент не використав інструменти web_search та python_repl_tool, хоча вони були передбачені для збору зовнішніх даних і виконання регресійного аналізу. У результаті частина висновків ґрунтується на припущеннях моделі, а не на фактичних даних або обчисленнях.
Щоб покращити роботу агента, варто:
- посилити системний промпт, щоб інструменти використовувалися обов’язково, а не опціонально;
- додати маршрутизацію в LangGraph, яка примусово спрямовує запит до Python‑аналізу або веб‑пошуку залежно від змісту;
- впровадити перевірку використання інструментів, щоб уникнути галюцинацій;
- розширити фінальну відповідь, включивши сценарний аналіз або діапазон прогнозів, обґрунтований реальними даними.
Такі покращення зроблять агента більш надійним, відтворюваним і корисним для бізнес‑аналітики.